# 03 — Campaign Analytics EDA
## Bank Marketing Dataset Analysis
**Important Dataset Limitation:** This dataset contains only 100 records,
all from May, all via telephone contact. This is a subset of the full UCI
Bank Marketing dataset. Analysis is constrained to what this data supports.
Full analysis would require the complete 41,188-record dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sqlite3
from datetime import datetime
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ── Load Dataset: Bank Marketing Campaign ─────────────────────────────────
df = pd.read_csv('../../data/raw/campaign/bank_campaign.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\ny distribution:")
print(df['y'].value_counts())
print(f"\nSuccess rate: {(df['y']=='yes').mean()*100:.1f}%")
print(f"\nUnique contact methods: {df['contact'].unique().tolist()}")
print(f"Unique months: {df['month'].unique().tolist()}")
print(f"\nDataset limitation: {len(df)} rows only — all {df['contact'].iloc[0]}, all {df['month'].iloc[0]}")
print(f"\nNull counts:")
print(df.isnull().sum())
print(f"\npdays value counts (999 = never contacted):")
print(df['pdays'].value_counts().head(10))

Shape: (100, 22)
Columns: ['index', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

y distribution:
y
no     97
yes     3
Name: count, dtype: int64

Success rate: 3.0%

Unique contact methods: ['telephone']
Unique months: ['may']

Dataset limitation: 100 rows only — all telephone, all may

Null counts:
index             0
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

pdays value counts (999 = never contacted):
p

In [3]:
# ── SQL Analysis on Campaign Data ───────────────────────────────────────
con = sqlite3.connect(':memory:')
df.to_sql('campaign', con, index=False, if_exists='replace')

# Q1: Success by job
q1 = pd.read_sql_query("""
    SELECT job, COUNT(*) as total,
           SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as converted,
           ROUND(100.0*SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END)/COUNT(*),2) as conv_pct
    FROM campaign GROUP BY job ORDER BY conv_pct DESC
""", con)
print("Q1: Success by Job")
print(q1.to_string(index=False))
print()

# Q2: Success by age group
q2 = pd.read_sql_query("""
    SELECT
      CASE WHEN age < 30 THEN 'Under 30'
           WHEN age BETWEEN 30 AND 45 THEN '30-45'
           WHEN age BETWEEN 46 AND 60 THEN '46-60'
           ELSE 'Over 60' END as age_group,
      COUNT(*) as total,
      SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as converted,
      ROUND(100.0*SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END)/COUNT(*),2) as conv_pct
    FROM campaign GROUP BY age_group ORDER BY conv_pct DESC
""", con)
print("Q2: Success by Age Group")
print(q2.to_string(index=False))
print()

# Q3: Success by poutcome (previous campaign outcome)
q3 = pd.read_sql_query("""
    SELECT poutcome, COUNT(*) as total,
           SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as converted
    FROM campaign GROUP BY poutcome
""", con)
print("Q3: Success by Previous Outcome")
print(q3.to_string(index=False))
print()

# Q4: Duration analysis — successful vs failed calls
q4 = pd.read_sql_query("""
    SELECT y, ROUND(AVG(duration),2) as avg_duration_seconds,
           ROUND(MIN(duration),2) as min_dur, ROUND(MAX(duration),2) as max_dur
    FROM campaign GROUP BY y
""", con)
print("Q4: Call Duration by Outcome")
print(q4.to_string(index=False))
print()

# Q5: Campaign contact count vs conversion
q5 = pd.read_sql_query("""
    SELECT campaign,
           COUNT(*) as total,
           SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as converted
    FROM campaign GROUP BY campaign ORDER BY campaign
""", con)
print("Q5: Campaign Contact Count vs Conversion")
print(q5.to_string(index=False))
print()

# Q6: Previous contacts analysis
q6 = pd.read_sql_query("""
    SELECT previous, COUNT(*) as count,
           SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as converted
    FROM campaign GROUP BY previous ORDER BY previous
""", con)
print("Q6: Previous Contacts Analysis")
print(q6.to_string(index=False))
print()

# Q7: Macroeconomic features vs conversion
q7 = pd.read_sql_query("""
    SELECT y,
           ROUND(AVG("emp.var.rate"),4) as avg_emp_var,
           ROUND(AVG("cons.price.idx"),4) as avg_cons_price,
           ROUND(AVG("euribor3m"),4) as avg_euribor,
           ROUND(AVG("nr.employed"),2) as avg_employed
    FROM campaign GROUP BY y
""", con)
print("Q7: Macroeconomic Context by Outcome")
print(q7.to_string(index=False))
print()

Q1: Success by Job
          job  total  converted  conv_pct
 entrepreneur      3          1   33.3300
   technician     15          1    6.6700
  blue-collar     24          1    4.1700
      unknown      4          0    0.0000
   unemployed      3          0    0.0000
     services     10          0    0.0000
self-employed      1          0    0.0000
      retired      2          0    0.0000
   management      9          0    0.0000
    housemaid      3          0    0.0000
       admin.     26          0    0.0000

Q2: Success by Age Group
age_group  total  converted  conv_pct
    46-60     46          2    4.3500
    30-45     49          1    2.0400
 Under 30      5          0    0.0000

Q3: Success by Previous Outcome
   poutcome  total  converted
nonexistent    100          3

Q4: Call Duration by Outcome
  y  avg_duration_seconds   min_dur   max_dur
 no              266.5300   20.0000 1666.0000
yes             1361.3300 1042.0000 1575.0000

Q5: Campaign Contact Count vs Convers

### ⚠️ Dataset Limitation

This dataset (100 rows) **cannot** support:
- Channel comparison (all telephone)
- Monthly trend analysis (all May)
- Statistically significant segment analysis (n=3 conversions only)

For production deployment, replace with full UCI Bank Marketing dataset
(41,188 rows from: https://archive.ics.uci.edu/dataset/222/bank+marketing)

**What this data CAN show:**
- Call duration is the strongest predictor (longer = more interested)
- Previous campaign success (`poutcome=success`) strongly predicts current success
- Macroeconomic context (`euribor3m`, `emp.var.rate`) matters

In [4]:
# ── Plot 1: Campaign Response Distribution — Pie Chart ────────────────
y_counts = df['y'].value_counts().reset_index()
y_counts.columns = ['Response', 'Count']
fig1 = px.pie(
    y_counts, names='Response', values='Count',
    hole=0.4,
    color='Response',
    color_discrete_map={'no': '#C0001A', 'yes': '#006FCF'},
    title='Campaign Response Distribution (100-record sample)<br><sub>Small sample — only 3 conversions</sub>'
)
fig1.update_traces(textposition='inside', textinfo='percent+label+value')
fig1.update_layout(template='plotly_white', height=450)
fig1.show()

In [5]:
# ── Plot 2: Call Duration by Campaign Outcome — Box Plot ─────────────
fig2 = px.box(
    df, x='y', y='duration', color='y',
    color_discrete_map={'no': '#C0001A', 'yes': '#006FCF'},
    title='Call Duration by Campaign Outcome (Key Predictor)<br>'
          '<sub>NOTE: duration is known only after call ends — post-hoc feature, not usable for prediction</sub>',
    template='plotly_white',
    labels={'y': 'Campaign Outcome', 'duration': 'Call Duration (seconds)'}
)
fig2.update_layout(height=450, showlegend=False)
fig2.show()

In [6]:
# ── Plot 3: Customer Age Distribution by Campaign Response ──────────
fig3 = px.histogram(
    df, x='age', color='y', barmode='overlay',
    opacity=0.7, nbins=20,
    color_discrete_map={'no': '#C0001A', 'yes': '#006FCF'},
    title='Customer Age Distribution by Campaign Response',
    template='plotly_white',
    labels={'age': 'Customer Age', 'y': 'Response'}
)
fig3.update_layout(height=400, xaxis_title='Age', yaxis_title='Count')
fig3.show()

In [7]:
# ── Plot 4: Customer Job Distribution — Bar Chart ───────────────────
job_counts = df['job'].value_counts().reset_index()
job_counts.columns = ['Job', 'Count']
fig4 = go.Figure()
fig4.add_trace(go.Bar(
    y=job_counts['Job'], x=job_counts['Count'],
    orientation='h', marker_color='#006FCF',
    text=job_counts['Count'], textposition='outside',
    name='Count'
))
fig4.update_layout(
    title='Customer Job Distribution in Campaign Dataset',
    xaxis_title='Count', yaxis_title='Job',
    template='plotly_white', height=500,
    margin=dict(l=150), yaxis=dict(autorange='reversed')
)
fig4.show()

In [8]:
# ── Plot 5: Number of Contacts Made in This Campaign ───────────────
fig5 = px.histogram(
    df, x='campaign', nbins=15,
    title='Number of Contacts Made in This Campaign',
    template='plotly_white',
    color_discrete_sequence=['#006FCF'],
    labels={'campaign': 'Number of Contacts', 'count': 'Frequency'}
)
fig5.update_layout(height=400, xaxis_title='Number of Contacts', yaxis_title='Count')
fig5.show()

In [9]:
# ── Plot 6: Macroeconomic Context at Time of Contact ───────────────
fig6 = px.scatter(
    df, x='euribor3m', y='cons.conf.idx', color='y',
    color_discrete_map={'no': '#C0001A', 'yes': '#006FCF'},
    title='Macroeconomic Context at Time of Contact',
    template='plotly_white',
    labels={'euribor3m': 'Euribor 3-Month Rate', 'cons.conf.idx': 'Consumer Confidence Index', 'y': 'Response'},
    size='duration', size_max=15, opacity=0.7
)
fig6.update_layout(height=450)
fig6.show()

In [10]:
# ── Feature Engineering for Model Use ───────────────────────────────
df['pdays_contacted'] = (df['pdays'] != 999).astype(int)
df['y_binary'] = (df['y'] == 'yes').astype(int)
df['duration_min'] = df['duration'] / 60
print(f"Previously contacted: {df['pdays_contacted'].sum()} out of {len(df)}")
print(f"\ny_binary distribution:")
print(df['y_binary'].value_counts())
print(f"\nDuration (minutes) stats:")
print(df['duration_min'].describe().round(2))
print(f"\nEngineered columns added: pdays_contacted, y_binary, duration_min")

Previously contacted: 0 out of 100

y_binary distribution:
y_binary
0    97
1     3
Name: count, dtype: int64

Duration (minutes) stats:
count   100.0000
mean      4.9900
std       4.6300
min       0.3300
25%       2.8700
50%       3.6500
75%       5.7200
max      27.7700
Name: duration_min, dtype: float64

Engineered columns added: pdays_contacted, y_binary, duration_min


In [11]:
# ── Save cleaned data ───────────────────────────────────────────────
df.to_csv('../../data/processed/campaign_clean.csv', index=False)
print(f"Saved campaign_clean.csv — shape {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("WARNING: Only 100 records. ML model on this data will have very high variance.")

con.close()

Saved campaign_clean.csv — shape (100, 25)
Columns: ['index', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y', 'pdays_contacted', 'y_binary', 'duration_min']
